# LLMs con Hugging Face: Local y Remoto

En el notebook anterior usamos Ollama para correr un LLM completamente en nuestra máquina.
Hugging Face nos abre tres caminos distintos para trabajar con modelos de lenguaje:

1. **Local con `transformers`** — el modelo se descarga y corre en tu computador
2. **Remoto con `InferenceClient`** — el modelo corre en los servidores de Hugging Face
3. **Comparativa** — ¿cuándo usar cada opción?

En este notebook usaremos la familia **Qwen2.5** de Alibaba, una de las mejores familias
open-source disponibles hoy. Esto también nos permite hablar de qué hace a un modelo "bueno".

## ¿Qué es Hugging Face?

Hugging Face es la plataforma de referencia para modelos open-source de IA. Ofrece:
- Un repositorio con cientos de miles de modelos listos para usar
- La biblioteca `transformers` para correr modelos localmente
- Un servicio de inferencia remota (Serverless Inference) con free tier
- Datasets, spaces (demos) y mucho más

## ¿Qué es Qwen2.5?

Qwen2.5 es una familia de modelos desarrollada por Alibaba Cloud, disponible en múltiples tamaños:
- **1.5B** parámetros: pequeño, corre en CPU, ideal para demos locales
- **7B, 14B, 32B**: balance entre calidad y recursos
- **72B** parámetros: calidad comparable a modelos propietarios, requiere GPU potente (o usarlo vía API)

Los modelos con sufijo `-Instruct` están afinados para seguir instrucciones en formato chat,
lo que los hace adecuados para construir asistentes conversacionales.

## Prerrequisitos

### Token de Hugging Face

Para usar el Serverless Inference (sección 2) necesitas un token gratuito:
1. Crea una cuenta en [huggingface.co](https://huggingface.co)
2. Ve a Settings → Access Tokens → New Token
3. Crea un token con permisos de lectura (Read)
4. Guárdalo como variable de entorno: `HF_TOKEN`

Para la sección local (sección 1) el token no es estrictamente necesario, pero sí recomendable
para evitar límites de descarga.
"""

# ============================================================
# SECCIÓN 1: LLM LOCAL CON TRANSFORMERS
# ============================================================

## 1. LLM Local con `transformers`

En esta sección descargamos el modelo `Qwen2.5-1.5B-Instruct` directamente a nuestra máquina
y lo ejecutamos localmente, sin ninguna llamada a internet en tiempo de inferencia.

Esto es conceptualmente igual a Ollama del notebook anterior: el modelo corre en tu hardware.
La diferencia es que aquí usamos directamente la biblioteca `transformers` de Hugging Face,
sin intermediarios.

![Alt text](figs/01/huggingface_local_architecture.svg) 

### Instalación
"""

In [ ]:
cd "/content/drive/MyDrive/Courses/AI_IS/GenAI"

In [ ]:
%%bash
#pip install transformers accelerate python-dotenv

In [ ]:
%%bash 
git clone https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

### 1.1 Cargando el Modelo

La primera vez que ejecutes esto, `transformers` descargará el modelo (~3GB).
Las veces siguientes lo cargará desde la caché local.

In [2]:
import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
# Nombre del modelo en Hugging Face Hub.
# Formato: "organizacion/nombre-del-modelo"
MODEL_ID = "Qwen2.5-1.5B-Instruct"

print(f"Cargando modelo: {MODEL_ID}")
print("La primera vez puede tardar varios minutos (descarga ~3GB)...")

# Cargamos el tokenizador: convierte texto en tokens (numeros) que el modelo entiende
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Cargamos el modelo.
# torch_dtype=torch.float16 reduce el uso de memoria a la mitad respecto a float32
# device_map="auto" detecta automaticamente si hay GPU disponible, si no usa CPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)

print("Modelo cargado exitosamente!")
print(f"Dispositivo en uso: {next(model.parameters()).device}")

### 1.2 Primera Inferencia

Con el modelo cargado, podemos generar texto. El flujo es:
1. Formatear los mensajes segun el template del modelo
2. Tokenizar (convertir texto a numeros)
3. Generar (el modelo predice los siguientes tokens)
4. Decodificar (convertir numeros a texto)


In [ ]:
# Definimos los mensajes en formato chat (igual que con Ollama o OpenAI)
messages = [
    {"role": "system", "content": "Eres un asistente util."},
    {"role": "user", "content": "Explica que es una red neuronal en dos oraciones."}
]

# apply_chat_template formatea los mensajes segun el formato especifico de Qwen2.5.
# tokenize=False devuelve el texto formateado antes de tokenizar, util para inspeccion.
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True  # agrega el marcador que indica que el modelo debe responder
)

# Tokenizamos y enviamos al dispositivo donde esta el modelo (CPU o GPU)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generamos la respuesta. max_new_tokens limita la longitud de la respuesta.
with torch.no_grad():  # desactivamos gradientes para ahorrar memoria en inferencia
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256
    )

# Extraemos solo los tokens nuevos (la respuesta), descartando el prompt de entrada
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Decodificamos los tokens a texto legible
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

### 1.3 Funcion Reutilizable

El proceso anterior tiene varios pasos. Vamos a encapsularlo en una funcion limpia
que usaremos en el resto de la seccion:

In [5]:
def chat_local(messages: list, max_new_tokens: int = 512) -> str:
    """
    Genera una respuesta usando el modelo local.

    Parametros:
        messages: lista de dicts con claves 'role' y 'content'
        max_new_tokens: longitud maxima de la respuesta en tokens

    Retorna:
        La respuesta del modelo como string
    """
    # Aplicamos el template de chat del modelo
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenizamos
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generamos
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens
        )

    # Extraemos y decodificamos solo la respuesta nueva
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
# Probamos la funcion con un prompt de codigo
messages = [
    {"role": "system", "content": "Eres un experto en Python."},
    {"role": "user", "content": "Escribe una funcion para verificar si un numero es primo."}
]

respuesta = chat_local(messages)
print(respuesta)

### 1.4 Chatbot Local con Memoria

Igual que en el notebook anterior, mantenemos el historial de conversacion
para que el modelo recuerde el contexto entre turnos:
"""

In [8]:
# Inicializamos el historial con el mensaje de sistema
historial_local = [
    {"role": "system", "content": "Eres un asistente util y conciso."}
]

def chat_local_con_memoria(user_input: str) -> str:
    """Chat con el modelo local manteniendo el historial de conversacion."""

    # Agregamos el mensaje del usuario al historial
    historial_local.append({"role": "user", "content": user_input})

    # Generamos la respuesta pasando todo el historial
    respuesta = chat_local(historial_local)

    # Mostramos el intercambio
    print(f"Tu: {user_input}")
    print(f"Asistente (Qwen2.5-1.5B local): {respuesta}")
    print(f"[Mensajes en historial: {len(historial_local) + 1}]")

    # Guardamos la respuesta en el historial para la proxima interaccion
    historial_local.append({"role": "assistant", "content": respuesta})

    return respuesta

In [ ]:
# Primera interaccion
chat_local_con_memoria("Mi color favorito es el azul.")

In [ ]:
# Segunda interaccion: recuerda?
chat_local_con_memoria("Cual es mi color favorito?")

### 1.5 La Forma Práctica: `pipeline`

Ahora que entendemos el proceso completo (tokenizar → generar → decodificar),
`transformers` ofrece una abstracción que hace todo eso en una sola línea: `pipeline`.
Es la forma recomendada cuando no necesitas control fino sobre el proceso de inferenci

In [11]:
from transformers import pipeline

In [18]:
# Creamos el pipeline de generación de texto
# Internamente hace exactamente lo mismo que hicimos en las secciones anteriores
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

In [ ]:
# Un solo llamado reemplaza todo el proceso manual
messages = [
    {"role": "system", "content": "Eres un asistente util."},
    {"role": "user", "content": "Explica que es una red neuronal en dos oraciones."}
]

output = pipe(messages, max_new_tokens=256)

# La respuesta viene en el último mensaje del historial generado
print(output[0]["generated_text"][-1]["content"])

### 1.6 Pipeline con Memoria

El pipeline acepta una lista de mensajes, así que podemos mantener
el historial exactamente igual que antes:

In [20]:
historial_pipeline = [
    {"role": "system", "content": "Eres un asistente util y conciso."}
]

def chat_pipeline(user_input: str) -> str:
    """Chat con pipeline manteniendo historial de conversacion."""

    # Agregamos el mensaje del usuario
    historial_pipeline.append({"role": "user", "content": user_input})

    # El pipeline recibe todo el historial
    output = pipe(historial_pipeline, max_new_tokens=256)

    # Extraemos solo la ultima respuesta generada
    respuesta = output[0]["generated_text"][-1]["content"]

    # Mostramos el intercambio
    print(f"Tu: {user_input}")
    print(f"Asistente: {respuesta}")
    print(f"[Mensajes en historial: {len(historial_pipeline) + 1}]")

    # Guardamos la respuesta en el historial
    historial_pipeline.append({"role": "assistant", "content": respuesta})

    return respuesta

In [ ]:
# Primera interaccion
chat_pipeline("Mi color favorito es el azul.")

In [ ]:
# Segunda interaccion: recuerda?
chat_pipeline("Cual es mi color favorito?")

## 2. LLM Remoto con `InferenceClient`

Ahora usaremos el modelo `Qwen2.5-72B-Instruct` corriendo en los servidores de HF.

Compara los números:
- Sección 1: **1.5B parámetros**, corre en tu GPU/CPU, ~3GB de VRAM/RAM
- Sección 2: **72B parámetros**, corre en servidores de HF, tú solo haces una llamada HTTP

La calidad de respuesta será notablemente superior. El tradeoff: necesitas internet,
dependes de un servicio externo, y el free tier tiene límites de uso.

![Alt text](figs/01/huggingface_remote_architecture.svg)    


### 2.1 Configuración del Cliente

In [24]:
from huggingface_hub import InferenceClient

In [ ]:
# Inicializamos el cliente — sin provider, el router de HF lo maneja automáticamente
client = InferenceClient(
    api_key=os.getenv("HF_ACCESS_TOKEN"),
)

MODELO_REMOTO = "Qwen/Qwen2.5-Coder-32B-Instruct:nscale"

print("Cliente de inferencia remota inicializado correctamente")

### 2.2 Primera Llamada al Modelo Remoto
La interfaz de `InferenceClient` sigue el mismo formato de mensajes que OpenAI,
lo cual es un estandar de facto en la industria. Notaras que el codigo es casi identico
al de la seccion anterior, solo cambia de donde viene el modelo.
"""

In [ ]:
# Mismo formato de mensajes que en la sección local
response = client.chat.completions.create(
    model=MODELO_REMOTO,
    messages=[
        {
            "role": "user",
            "content": "Explica que es una red neuronal en dos oraciones"
        }
    ],
)

print(response.choices[0].message.content)

### 2.3 Funcion Reutilizable

Encapsulamos la llamada remota en una funcion con la **misma firma** que `chat_local`.
Esto es intencional: el patron de interaccion con un LLM es el mismo
independiente de donde corra el modelo.

In [ ]:
def chat_remoto(messages: list, max_tokens: int = 512) -> str:
    """
    Genera una respuesta usando el modelo remoto en HF Serverless Inference.

    Parametros:
        messages: lista de dicts con claves 'role' y 'content'
        max_tokens: longitud maxima de la respuesta en tokens

    Retorna:
        La respuesta del modelo como string
    """
    response = client.chat.completions.create(
        model=MODELO_REMOTO,
        messages=messages,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# Probamos con el mismo prompt de codigo de la seccion anterior
# Compara la calidad de respuesta entre 1.5B local y 72B remoto
messages = [
    {"role": "system", "content": "Eres un experto en Python."},
    {"role": "user", "content": "Escribe una funcion para verificar si un numero es primo."}
]

respuesta = chat_remoto(messages)
print(respuesta)

Nota la diferencia en calidad respecto a la version local. El modelo mas grande
tiende a escribir codigo mas limpio, con mejores comentarios y manejo de casos borde.

### 2.4 Chatbot Remoto con Memoria

In [67]:
# Historial separado para no mezclar con el historial local
historial_remoto = [
    {"role": "system", "content": "Eres un asistente util y conciso."}
]

def chat_remoto_con_memoria(user_input: str) -> str:
    """Chat con el modelo remoto manteniendo el historial de conversacion."""

    # Agregamos el mensaje del usuario
    historial_remoto.append({"role": "user", "content": user_input})

    # Llamamos al modelo remoto con todo el historial
    respuesta = chat_remoto(historial_remoto)

    # Mostramos el intercambio
    print(f"Tu: {user_input}")
    print(f"Asistente (Qwen2.5-72B remoto): {respuesta}")
    print(f"[Mensajes en historial: {len(historial_remoto) + 1}]")

    # Guardamos la respuesta en el historial
    historial_remoto.append({"role": "assistant", "content": respuesta})

    return respuesta


In [ ]:
# Primera interaccion
chat_remoto_con_memoria("Mi color favorito es el azul.")

In [ ]:
# Segunda interaccion: recuerda?
chat_remoto_con_memoria("Cual es mi color favorito?")

### 2.5 Streaming con el Modelo Remoto

El Serverless Inference tambien soporta streaming: el texto aparece
progresivamente en lugar de esperar a que la respuesta este completa.
Es la misma experiencia que ves en ChatGPT o Claude.

In [73]:
def chat_remoto_streaming(user_input: str) -> str:
    """Chat remoto con streaming: imprime la respuesta token por token."""

    messages = [
        {"role": "system", "content": "Eres un asistente util."},
        {"role": "user", "content": user_input}
    ]

    print(f"Tu: {user_input}")
    print("Asistente: ", end="", flush=True)

    respuesta_completa = ""

    # stream=True activa el modo streaming: el servidor envia chunks a medida que genera
    stream = client.chat.completions.create(
        model=MODELO_REMOTO,
        messages=messages,
        max_tokens=512,
        stream=True
    )

    # Iteramos sobre los chunks a medida que llegan del servidor
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content is not None:
            fragmento = chunk.choices[0].delta.content
            respuesta_completa += fragmento
            print(fragmento, end="", flush=True)

    print("\n")  # salto de linea al terminar
    return respuesta_completa

In [ ]:
# Probamos el streaming
chat_remoto_streaming("Escribe un poema corto sobre la inteligencia artificial.")

## 3. Comparativa: Cuando Usar Cada Opcion?

Hasta ahora hemos visto tres formas de correr un LLM:
- **Ollama** (notebook 01): modelo local, interfaz simplificada
- **HF Local** (seccion 1): modelo local, control total con `transformers`
- **HF Remoto** (seccion 2): modelo remoto, calidad superior, free tier

A estas tres se sumara en el proximo notebook la **API de Gemini** como ejemplo
de API propietaria. La siguiente tabla resume los tradeoffs clave:

| Caracteristica       | Ollama (local) | HF Local (transformers) | HF Remoto (Serverless) | API Propietaria (Gemini, OpenAI) |
|----------------------|----------------|-------------------------|------------------------|----------------------------------|
| Donde corre el modelo | Tu maquina    | Tu maquina              | Servidores de HF       | Servidores del proveedor         |
| Privacidad de datos  | Total          | Total                   | Datos salen de tu maquina | Datos en servidores externos  |
| Costo                | Gratis         | Gratis                  | Free tier disponible   | Pago por token                   |
| Calidad del modelo   | Limitada por tu hardware | Limitada por tu hardware | Alta (modelos grandes) | Muy alta               |
| Requiere internet    | No             | No                      | Si                     | Si                               |
| Control del modelo   | Total          | Total                   | Parcial                | Ninguno                          |
| Facilidad de uso     | Alta           | Media                   | Alta                   | Alta                             |
| Escalabilidad        | Limitada       | Limitada                | Limitada (free tier)   | Alta                             |
| Modelos open-source  | Si             | Si                      | Si                     | No (propietarios)                |

### Cuando usar cada uno?

**Ollama / HF Local:** cuando los datos son sensibles y no pueden salir de tu maquina
(salud, finanzas, datos privados), en entornos sin internet, o cuando necesitas
experimentar con los pesos del modelo directamente.

**HF Remoto (Serverless):** prototipado rapido con modelos grandes sin GPU propia,
proyectos academicos con presupuesto limitado, o cuando necesitas mas calidad que
un modelo local pequeno sin incurrir en costos.

**APIs Propietarias (Gemini, OpenAI, Claude):** produccion con alta disponibilidad,
cuando necesitas los modelos mas capaces del mercado, o cuando el costo por token
es aceptable para el caso de uso.

### Una reflexion sobre open-source vs propietario

El ecosistema open-source ha avanzado enormemente. Modelos como Qwen2.5-72B,
Llama 3.1-70B o Mistral Large son competitivos con GPT-4 en muchas tareas.
La brecha se ha reducido significativamente desde 2023.

La eleccion entre open-source y propietario hoy no es solo de calidad, sino de:
- **Control**: necesitas modificar o inspeccionar el modelo?
- **Privacidad**: los datos pueden salir de tu infraestructura?
- **Costo**: el volumen de uso hace viable pagar por token?
- **Compliance**: hay restricciones legales o regulatorias?

## 4. Recursos para Seguir Aprendiendo

- [Hugging Face Hub](https://huggingface.co/models) - catalogo de modelos
- [Documentacion de transformers](https://huggingface.co/docs/transformers)
- [Documentacion de InferenceClient](https://huggingface.co/docs/huggingface_hub/guides/inference)
- [Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard)
- [Qwen2.5 en HF](https://huggingface.co/collections/Qwen/qwen25-66e81a666513e518adb90d9e)